# 03 · Search

The recipe is always the same:

1. Embed your collection with `embed_documents`.
2. Embed the question with `embed_queries`, using the same task.
3. `eg.top_k` gives the best matches.

This notebook runs that recipe over text, images, videos, audio and mixed items.

In [1]:
import os
from pathlib import Path

import embeddinggemma_engine as eg

if Path.cwd().name == "notebooks":
    os.chdir("..")  # run from the repo root, so paths stay short
IMAGES = Path("assets/images")
VIDEOS = Path("assets/videos")
AUDIO = Path("assets/audio")


def mistake(fn):
    """Run fn and print the error the engine raises for it."""
    try:
        fn()
    except eg.EngineError as error:
        print(f"{type(error).__name__}: {error}\n")

In [2]:
engine = eg.EmbeddingEngine()


def search(query, documents, names, k=3, task="search"):
    """Embed one query and print the k best documents."""
    hits = eg.top_k(engine.embed_queries([query], task=task), documents, k=k)[0]
    print(query)
    for hit in hits:
        print(f"   {hit.score:.2f}  {names[hit.index]}")

## Text
A small knowledge base with titles.

In [3]:
notes = {
    "Naan at home": "A hot cast-iron pan works almost as well as a tandoor: roll the dough thin and cook it dry.",
    "Pour-over coffee": "Bloom the grounds for thirty seconds, then pour slowly in circles.",
    "Why zebras have stripes": "Stripes seem to deter biting flies, which struggle to land on striped surfaces.",
    "Coral reefs": "Reefs cover under one percent of the ocean floor but shelter a quarter of marine species.",
    "Tyre dumps": "Discarded tyres trap rainwater, breed mosquitoes and can burn for months.",
}
note_vectors = engine.embed_documents([eg.Item.text(text, title=title) for title, text in notes.items()],
                                      task="search")

search("how can I make naan without a tandoor?", note_vectors, list(notes))
search("why are zebras striped", note_vectors, list(notes))

how can I make naan without a tandoor?
   0.84  Naan at home
   0.57  Tyre dumps
   0.54  Pour-over coffee
why are zebras striped
   0.79  Why zebras have stripes
   0.56  Tyre dumps
   0.55  Coral reefs


A more specific task can be used when it fits, for example `question_answering` for questions.
Remember to embed the documents with that same task.

In [4]:
qa_vectors = engine.embed_documents([eg.Item.text(text, title=title) for title, text in notes.items()],
                                    task="question_answering")
search("What share of sea life lives on reefs?", qa_vectors, list(notes), task="question_answering")

What share of sea life lives on reefs?
   0.83  Coral reefs
   0.56  Why zebras have stripes
   0.56  Tyre dumps


## Images

In [5]:
photos = ["wildlife.jpg", "trash.jpg", "dishes.jpg", "aquatic.jpg", "kitchen.jpg", "merchandise.jpg"]
photo_vectors = engine.embed_documents([eg.Item.image(IMAGES / p) for p in photos], task="search")

search("giraffes and zebras", photo_vectors, photos, k=2)
search("curry dinner", photo_vectors, photos, k=2)

giraffes and zebras
   0.78  wildlife.jpg
   0.62  aquatic.jpg
curry dinner
   0.69  dishes.jpg
   0.58  merchandise.jpg


## Videos
Each video becomes one vector, built from one frame per second.

In [6]:
clips = ["01_dog.mp4", "02_cups.mp4", "03_blocks.mp4", "04_coffee.mp4", "05_juggle.mp4"]
clip_vectors = engine.embed_documents([eg.Item.video(VIDEOS / c) for c in clips], task="search")

search("someone playing with a puppy", clip_vectors, clips, k=2)
search("football tricks", clip_vectors, clips, k=2)

someone playing with a puppy
   0.67  01_dog.mp4
   0.59  03_blocks.mp4
football tricks
   0.67  05_juggle.mp4
   0.61  02_cups.mp4


## Audio
Audio works the same way. Text→audio is this model's weakest pairing, though: the scores sit
close together, so look at the gap between results before trusting the top one.

In [7]:
sounds = {
    "soda can": "voicebosch-soda-can-opening-167415.mp3",
    "notification": "musicwallah-soft_notification-sfx-567244.mp3",
    "halloween": "dragon-studio-halloween-creepy-stock-sound-effect-511328.mp3",
    "flamenco": "ahmadmousavipour-1throw-the-chord-flamenco-stock-299645.mp3",
    "corporate music": "joyinsound-corporate-upbeat-motivational-music-royalty-free-496474.mp3",
}
sound_vectors = engine.embed_documents([eg.Item.audio(AUDIO / f) for f in sounds.values()], task="search")

search("opening a fizzy drink", sound_vectors, list(sounds), k=3)

opening a fizzy drink
   0.64  soda can
   0.64  notification
   0.57  halloween


## Mixed collections
Text, images and video can sit in **one** collection, so one query searches all of them.
Items can also mix parts themselves, like a listing with a description and a photo.

In [8]:
mixed = [
    eg.Item.text("Learn to brew pour-over coffee with a gooseneck kettle.", title="Coffee class"),
    eg.Item.of("Safari day trip.", eg.Image(IMAGES / "wildlife.jpg"), title="Savanna tour"),
    eg.Item.video(VIDEOS / "05_juggle.mp4"),
    eg.Item.image(IMAGES / "aquatic.jpg"),
]
mixed_names = ["coffee class (text)", "savanna tour (text + photo)", "juggling (video)", "aquarium (photo)"]
mixed_vectors = engine.embed_documents(mixed, task="search")

search("see wild animals", mixed_vectors, mixed_names, k=2)
search("fish", mixed_vectors, mixed_names, k=2)

see wild animals
   0.70  savanna tour (text + photo)
   0.64  aquarium (photo)
fish
   0.73  aquarium (photo)
   0.61  savanna tour (text + photo)


## Smaller vectors
`dim=256` (or 512, 128) stores vectors 3–6× smaller, at a small cost in quality.
Queries and documents must use the same `dim`.

In [9]:
small_docs = engine.embed_documents([eg.Item.image(IMAGES / p) for p in photos], task="search", dim=256)
hits = eg.top_k(engine.embed_queries(["giraffes and zebras"], task="search", dim=256), small_docs, k=1)
print(photos[hits[0][0].index], f"{hits[0][0].score:.2f}", "| bytes per vector:", small_docs.vectors[0].nbytes)

wildlife.jpg 0.80 | bytes per vector: 1024


## Common mistakes

In [10]:
print("A file path written as a plain string is text, not an image:")
print("  ", eg.preview("assets/images/kitchen.jpg", task="search", side="document"))
print("   -> wrap it: eg.Item.image(path)\n")

print("Passing one item instead of a list:")
mistake(lambda: engine.embed_queries("fish", task="search"))

print("Query and documents with different dims:")
mistake(lambda: eg.top_k(engine.embed_queries(["fish"], task="search", dim=768), small_docs, k=1))

A file path written as a plain string is text, not an image:
   title: none | text: assets/images/kitchen.jpg
   -> wrap it: eg.Item.image(path)

Passing one item instead of a list:
InputError: Pass a list of items, e.g. [item], not a single item.

Query and documents with different dims:
IncompatibleEmbeddingsError: Cannot compare these embeddings: dim 768 vs 256

